In [4]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

# 验证
gpu_count = torch.cuda.device_count()
print(f"PyTorch can see {gpu_count} GPUs.")

PyTorch can see 4 GPUs.


In [ ]:
from zai import ZhipuAiClient
from openai import OpenAI
import base64

# client = ZhipuAiClient(api_key="bb4d71f420554dbcb4751c1220b6d49e.LkKhpmPLO8lH0hdp")  

DASHSCOPE_API_KEY = "sk-0r2nJC485Sq6MXAXGqTsrPPQccBTIdperwxk10ze72pGyhm8"
BASE_URL = "https://api.lingyaai.cn/v1"

MODEL_NAME = "gpt-5.4" 

client = OpenAI(
        api_key=DASHSCOPE_API_KEY,
        base_url=BASE_URL
    )

video_path = "mustardpp.mp4" 
with open(video_path, "rb") as video_file:
    video_base64 = base64.b64encode(video_file.read()).decode('utf-8')

response = client.chat.completions.create(
    model= MODEL_NAME,  
    messages=[
        {   
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_base64
                    }
                },
                {
                    "type": "text",
                    "text":         
        # f"""
        # You are a video-based Gaze Analysis Expert.
        # Task: Determine whether the speaker shows (1) a contemptuous squint / side-eye or (2) a eye-roll  at any time in the video.
        # Procedure:
        # 1.	Scan the full video and list up to 3 candidate moments with timestamps (mm:ss–mm:ss).
        # 2.	For each moment, describe objective cues only (gaze direction, eyelid aperture, head yaw/pitch, duration).
        # 3.	Decide: Squint/side-eye present? Eye-roll present? 
        # Output format (no extra text):
        # evidence:[ type: “eye_roll”|“side_eye_squint”, start:“mm:ss”, end:“mm:ss”, cues:”…” ]
        # eye_roll: Yes/No
        # side_eye_squint: Yes/No
        # final: Yes if either is Yes, else No
    
        # """

# f"""
#     You are a Human Behavior Analysis Expert specialized in detecting subtle non-verbal cues.
#     I have detected a potential 'side_eye' signal in the red bounding box. 
#     Your task is to VERIFY if this detection is a genuine social signal or a false positive (e.g., normal looking around, blinking).

#     Please analyze the video focusing on the red bounding box:
#     1. **Head vs. Eye Direction**: Does the person's head rotate with the eyes? (For side-eye, the head stays still while eyes move to the corner. For eye-roll, the eyes move upward/circularly independent of head motion).
#     (note that the eye-roll is usually mixed with constant downward gaze or closing of the eyes)
#     2. **Sclera Visibility**: Can you see a significant amount of the white part (sclera) of the eye?

#     Based on the above, answer with the following format:
#     - Observation: [Describe the eye and head movement briefly]
#     - Verdict: [True Positive / False Positive]
#     """

"""describe this video clip and recognize if howard is sarcastic?
I am happy to report I'm feeling much better.	0:02.867000	SHELDON	BBT
Good for you.	0:05.537000	LEONARD	BBT
My fever is gone, my sinuses are pressure-free,	0:08.306000	SHELDON	BBT
and my mucus is as clear as a Yosemite waterfall.	0:11.643000	SHELDON	BBT
Glad to hear it.	0:3.503	HOWARD	BBT"""
                }
            ],
            "role": "user"
        }
    ],
    # thinking={
    #     "type":"enabled"
    # }
)
print(response.choices[0].message.content.strip())

        
## 之前让GLM描述gaze的prompt
        # prompt = f"""
        # You are a Gaze Analysis Expert.
        # The provided video captures the main speaker [{speaker_name}] during a conversation.
        # Your task is to synthesize the visual changes of [{speaker_name}] into a single gaze behavior summary.

        # # Controlled Vocabulary (Strict Usage)
        # - Events: Use ONLY "fixation" (steady gaze), "saccade" (rapid shift), or "smooth pursuit" (tracking motion).
        # - Modifiers: Use "brief/long" for duration, "small/large" for amplitude, "up/down/left/right" for direction.
        # - Blinks: Quick (normal blink), Extended (long blink/closure), Rapid-fire (multiple blinks in short succession).

        # # Output Constraints
        # - Maximum 40 words.
        # - NO numbers, NO coordinates, NO technical explanations.
        # - Focus strictly on eye movements and blinks.

        # Describe the gaze sequence now.
        # """

**Clip description (what’s happening):**  
Sheldon announces that he’s feeling much better after being sick. He lists detailed, slightly gross medical improvements (“fever is gone,” “sinuses are pressure-free,” and his “mucus is as clear as a Yosemite waterfall”). Leonard briefly responds supportively (“Good for you”). Howard then replies “Glad to hear it.”

**Is Howard being sarcastic?**  
Most likely **yes, mildly sarcastic / politely dismissive**. Sheldon’s vivid mucus description is awkward and oversharing, and Howard’s short “Glad to hear it” reads like a quick, socially obligatory response rather than genuine interest. In context of *The Big Bang Theory*, Howard often reacts to Sheldon’s TMI with dry, understated sarcasm.

**But:** with only the text (no tone/voice), it could also be read as neutral politeness. The sarcasm interpretation depends heavily on Howard’s delivery (flat tone, slight emphasis) and the characters’ typical dynamic.


# 直接GLM判断讽刺

In [9]:
import json
import os
import argparse
import base64
from typing import List
from tqdm import tqdm
from zai import ZhipuAiClient 
from openai import OpenAI

# ================= 配置区域 =================
# API 配置
ZHIPU_API_KEY = "bb4d71f420554dbcb4751c1220b6d49e.LkKhpmPLO8lH0hdp" 
# GLM_MODEL_NAME = "glm-4.6v" 
GLM_MODEL_NAME = "glm-4.6v-flash" 

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/ablation_glm46v.jsonl" # 修改输出文件名以区分
# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

def infer_sarcasm_direct(video_path: str, speaker: str, utterance: str, context_str: str, client: ZhipuAiClient) -> dict:
    """
    直接将视频和文本上下文发送给 GLM 进行端到端讽刺检测
    """
    if not os.path.exists(video_path):
        return {"sarcasm": False, "reasoning": "Video not found", "error": True}

    try:
        # 1. 视频转 Base64
        with open(video_path, "rb") as video_file:
            video_base64 = base64.b64encode(video_file.read()).decode('utf-8')

        # 2. 构建简单粗暴的 Prompt
        prompt = f"""
        You are a Sarcasm Detection Expert analyzing a TV show clip.
        
        - Speaker: {speaker}
        
        ### Task:
        Watch the video and read the context. Determine if the speaker is sarcastic.
        Pay close attention to:
        1. Visual cues: Eye-rolling, side-eye, smirking, deadpan face, or exaggerated expressions.

        ### Output Format:
        Strictly return a JSON object (no markdown, no extra text):
        {{
            "analysis": "Briefly describe the visual expressions and why it is or isn't sarcastic in 1-2 sentences.",
            "sarcasm": true or false
        }}
        """

        # 3. 调用 API
        response = client.chat.completions.create(
            model=GLM_MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "video_url",
                            "video_url": {
                                "url": video_base64
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ],
            temperature=0.1, # 低温度保证输出格式稳定
            top_p=0.7
        )
        
        content = response.choices[0].message.content.strip()
        return parse_json_safely(content)

    except Exception as e:
        print(f"⚠️ API Error on {video_path}: {e}")
        return {"sarcasm": False, "reasoning": str(e), "error": True}

def parse_json_safely(content: str) -> dict:
    """
    清洗并解析 GLM 返回的内容，提取 JSON
    """
    try:
        # 尝试直接解析
        return json.loads(content)
    except:
        # 尝试去除 Markdown 代码块标记
        cleaned = content.replace("```json", "").replace("```", "").strip()
        try:
            return json.loads(cleaned)
        except:
            pass
            
        # 暴力截取 {} 之间的内容
        try:
            start = cleaned.find("{")
            end = cleaned.rfind("}") + 1
            if start != -1 and end != -1:
                return json.loads(cleaned[start:end])
        except:
            pass
            
    return {"sarcasm": False, "reasoning": "JSON Parsing Failed", "raw_response": content}

def main():
    args = parse_args()
    
    # 初始化客户端
    zhipu_client = ZhipuAiClient(api_key=ZHIPU_API_KEY)

    # 确保输出目录
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入
    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 断点续传逻辑
    processed_ids = set()
    if os.path.exists(args.output_file):
        with open(args.output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    res = json.loads(line)
                    if 'video_id' in res: 
                        processed_ids.add(str(res['video_id']))
                except:
                    pass

    print(f"🚀 开始处理 - 模型: {GLM_MODEL_NAME} (Direct Inference)")
    
    # 统一处理 list 或 dict 格式的数据
    items = data.items() if isinstance(data, dict) else enumerate(data)

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing"):
            
            # 获取 ID
            if isinstance(data, list):
                current_id = str(item.get('video_id', vid_id))
            else:
                current_id = str(vid_id)
            
            if current_id in processed_ids:
                continue

            # 视频路径
            video_file = f"{current_id}.mp4"
            video_path = os.path.join(args.video_folder, video_file)
            
            # 准备数据
            speaker = item.get("speaker", "Speaker")
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            # 拼接对话历史
            context_str = ""
            if context_list:
                for s, u in zip(context_speakers, context_list):
                    context_str += f"- {s}: {u}\n"
            else:
                context_str = "No previous context available."

            # ================= 核心推理 =================
            result_dict = infer_sarcasm_direct(video_path, speaker, utterance, context_str, zhipu_client)
            
            # 构造保存结果
            save_item = {
                "video_id": current_id,
                "ground_truth": item.get("sarcasm", None),
                # "utterance": utterance,
                "prediction_reasoning": result_dict.get("analysis", result_dict.get("reasoning", "")),
                "sarcasm_prediction": result_dict.get("sarcasm", False),
                "raw_response": result_dict.get("raw_response", "")
            }

            out_f.write(json.dumps(save_item, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 完成！结果保存至 {args.output_file}")

if __name__ == "__main__":
    main()

🚀 开始处理 - 模型: glm-4.6v-flash (Direct Inference)


Processing:   0%|          | 0/690 [00:00<?, ?it/s]

Processing:   7%|▋         | 49/690 [00:25<05:38,  1.90it/s]


KeyboardInterrupt: 

# 直接Qwen3VL/GPT/Gemini识别讽刺

In [20]:
import json
import os
import argparse
import base64
import cv2
import numpy as np
from typing import List
from tqdm import tqdm
from openai import OpenAI  

# ================= 配置区域 =================
# 阿里云
# DASHSCOPE_API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29" 
# BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"

# api
# DASHSCOPE_API_KEY = "sk-0r2nJC485Sq6MXAXGqTsrPPQccBTIdperwxk10ze72pGyhm8"
# BASE_URL = "https://api.lingyaai.cn/v1"

DASHSCOPE_API_KEY = "sk-3rEm8xqtTbrUzcYibkLGol6g5dVT59qWi9KERIyIkd8pE3C3"
BASE_URL = "https://jeniya.top/v1"

MODEL_NAME = "gemini-3.1-pro-preview" 
# MODEL_NAME = "qwen-vl-max" # 如果需要更强能力

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/ablation_gemini_vl.jsonl" 

# 抽帧配置
NUM_FRAMES = 8  # 抽取8帧传给模型（平衡成本和效果）
RESIZE_DIM = 512 # 缩放图片长边至512，加快传输和推理
# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

def extract_frames_base64(video_path: str, num_frames: int = NUM_FRAMES) -> List[str]:
    """
    使用 OpenCV 均匀抽取帧，并转为 Base64 字符串列表。
    这是 OpenAI 接口调用视觉模型的标准做法。
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    # 均匀采样索引
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    base64_frames = []

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        if i in indices:
            # 1. BGR 转 RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # 2. 缩放 (保持宽高比)
            h, w = frame.shape[:2]
            scale = RESIZE_DIM / max(h, w)
            new_w, new_h = int(w * scale), int(h * scale)
            frame = cv2.resize(frame, (new_w, new_h))

            # 3. 编码为 JPEG
            _, buffer = cv2.imencode('.jpg', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            
            # 4. 转 Base64
            b64_str = base64.b64encode(buffer).decode('utf-8')
            base64_frames.append(b64_str)
            
            if len(base64_frames) >= num_frames:
                break
    
    cap.release()
    return base64_frames

def infer_sarcasm_qwen(video_path: str, speaker: str, utterance: str, context_str: str, client: OpenAI) -> dict:
    """
    抽取视频帧 -> 发送给 Qwen-VL-Plus
    """
    if not os.path.exists(video_path):
        return {"sarcasm": False, "reasoning": "Video not found", "error": True}

    try:
        # 1. 抽取帧 (Qwen 无法直接接受 mp4 base64，必须传图片序列)
        frames_b64 = extract_frames_base64(video_path)
        if not frames_b64:
            return {"sarcasm": False, "reasoning": "Could not extract frames", "error": True}

        # 2. 构建 Prompt
        system_prompt = "You are a Sarcasm Detection Expert analyzing a TV show clip."
        user_text = f"""
        
        ### Task:
        Analyze the provided video frames and the context. Determine if the speaker is sarcastic.
        Look for: Eye-rolling, side-eye, smirking, deadpan delivery vs context.

        ### Output JSON:
        {{
            "analysis": "Reasoning in 1-2 sentences...",
            "sarcasm": true/false
        }}
        """

        # 3. 构建 OpenAI 格式的消息 payload
        # 混合文本和多张图片
        content_payload = [{"type": "text", "text": user_text}]
        
        # 将每一帧作为一个 image_url 添加
        for b64 in frames_b64:
            content_payload.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{b64}"
                }
            })

        # 4. 调用 API
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": content_payload}
            ],
            temperature=0.01, # Qwen 对 temperature 比较敏感，建议设低
        )
        
        content = response.choices[0].message.content.strip()
        return parse_json_safely(content)

    except Exception as e:
        print(f"⚠️ Qwen API Error on {video_path}: {e}")
        return {"sarcasm": False, "reasoning": str(e), "error": True}

def parse_json_safely(content: str) -> dict:
    """
    清洗并解析 Qwen 返回的内容
    """
    try:
        return json.loads(content)
    except:
        cleaned = content.replace("```json", "").replace("```", "").strip()
        try:
            return json.loads(cleaned)
        except:
            pass
        try:
            start = cleaned.find("{")
            end = cleaned.rfind("}") + 1
            if start != -1 and end != -1:
                return json.loads(cleaned[start:end])
        except:
            pass
            
    return {"sarcasm": False, "reasoning": "JSON Parsing Failed", "raw_response": content}

def main():
    args = parse_args()
    
    # ⚠️ 初始化 OpenAI 客户端 (指向阿里云 DashScope)
    client = OpenAI(
        api_key=DASHSCOPE_API_KEY,
        base_url=BASE_URL
    )

    # 确保输出目录
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入
    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 断点续传逻辑 (使用 raw_decode 读取以防止 JSONL 粘连问题)
    processed_ids = set()
    if os.path.exists(args.output_file):
        try:
            with open(args.output_file, 'r', encoding='utf-8') as f:
                file_content = f.read()
                decoder = json.JSONDecoder()
                pos = 0
                while pos < len(file_content):
                    while pos < len(file_content) and file_content[pos].isspace(): pos += 1
                    if pos >= len(file_content): break
                    try:
                        obj, end = decoder.raw_decode(file_content, idx=pos)
                        if 'video_id' in obj: processed_ids.add(str(obj['video_id']))
                        pos = end
                    except: pos += 1
        except Exception as e:
            print(f"⚠️ 读取旧文件出错 (可忽略): {e}")

    print(f"🚀 开始处理 - 模型: {MODEL_NAME} (Qwen-VL via OpenAI API)")
    
    items = data.items() if isinstance(data, dict) else enumerate(data)

    # 为了避免 OpenCV 占用显存，强制只使用 CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = "" 

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing"):
            
            if isinstance(data, list):
                current_id = str(item.get('video_id', vid_id))
            else:
                current_id = str(vid_id)
            
            if current_id in processed_ids:
                continue

            video_file = f"{current_id}.mp4"
            video_path = os.path.join(args.video_folder, video_file)
            
            # 数据准备
            speaker = item.get("speaker", "Speaker")
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            context_str = ""
            if context_list:
                for s, u in zip(context_speakers, context_list):
                    context_str += f"- {s}: {u}\n"
            else:
                context_str = "No previous context."

            # ================= 推理 =================
            result_dict = infer_sarcasm_qwen(video_path, speaker, utterance, context_str, client)
            
            save_item = {
                "video_id": current_id,
                "ground_truth": item.get("sarcasm", None),
                "sarcasm_prediction": result_dict.get("sarcasm", False),
                "utterance": utterance,
                "prediction_reasoning": result_dict.get("analysis", result_dict.get("reasoning", "")),
                "raw_response": result_dict.get("raw_response", "")
            }

            out_f.write(json.dumps(save_item, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 完成！结果保存至 {args.output_file}")

if __name__ == "__main__":
    main()

🚀 开始处理 - 模型: gemini-3.1-pro-preview (Qwen-VL via OpenAI API)


Processing:   0%|          | 0/690 [00:00<?, ?it/s]

Processing: 100%|██████████| 690/690 [5:26:19<00:00, 28.38s/it]    

✅ 完成！结果保存至 ./results/ablation_gemini_vl.jsonl


# Qwen3/GPT/Gemini + Pipeline (no phase3)

In [ ]:
import os
import cv2
import json
import math
import numpy as np
import pandas as pd
import mediapipe as mp
from typing import List, Dict, Tuple
import io
from PIL import Image
from tqdm import tqdm
import tempfile
import shutil
import re
from feat import Detector
from openai import OpenAI  
from tqdm import tqdm

# ================= 配置区域 =================
# API 配置 (使用 DashScope API Key)
# 获取地址: https://bailian.console.aliyun.com/
DASHSCOPE_API_KEY = "sk-bbf9f24ff4194920a43e15749a2dad29" 
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1" 

# xiaoji nongchang
# DASHSCOPE_API_KEY = "sk-1LhSpNg827CLx5oQZMDvDaGGkpj3ZeAFMCXWYZMpuuoSVKKR"
# BASE_URL = "https://api2.68886868.xyz/v1"

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url=BASE_URL
)

# LLM_MODEL_NAME = "[codex转] gpt-5.4"
# LLM_MODEL_NAME = "[不稳定渠道] gemini-3.1-pro"
LLM_MODEL_NAME = "qwen-plus-latest"
# LLM_MODEL_NAME = "qwen2.5:7b" 

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/ablation_no_phase3_naive.jsonl" 
CACHE_FILE = "./results/symbolic_reports_cache.json"

TARGET_FRAME_COUNT = 10 

# --- Phase 1 配置 ---
Z_SCORE_THRESH_MICRO = 2.0  
MIN_DURATION_SEC = 0.2      
ANALYSIS_FPS_LIMIT = 20.0  

# 将 face_mesh 声明为全局变量，改为懒加载（在需要时才初始化）
face_mesh = None

# =====================================================================
# Phase 1: Identity-Calibrated Symbolic Parser
# =====================================================================

def extract_visual_signals_optimized(video_path: str, detector: 'Detector') -> Dict[str, np.ndarray]:
    global face_mesh
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return {}
        
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if not original_fps or math.isnan(original_fps) or original_fps <= 0:
        original_fps = 25.0
        
    step = max(1, int(round(original_fps / ANALYSIS_FPS_LIMIT)))
    actual_fps = original_fps / step

    timestamps, gaze_y_list, gaze_x_list, frame_buffer = [], [], [],[]
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx % step == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            timestamps.append(frame_idx / original_fps)
            
            results = face_mesh.process(rgb_frame)
            if results.multi_face_landmarks:
                lm = results.multi_face_landmarks[0].landmark
                eye_h = abs(lm[145].y - lm[159].y) + 1e-6
                eye_w = abs(lm[133].x - lm[33].x) + 1e-6
                gaze_y_list.append((lm[145].y - lm[468].y) / eye_h)
                gaze_x_list.append((lm[468].x - lm[33].x) / eye_w)
            else:
                gaze_y_list.append(gaze_y_list[-1] if gaze_y_list else 0.5)
                gaze_x_list.append(gaze_x_list[-1] if gaze_x_list else 0.5)
            
            frame_buffer.append(frame) 

        frame_idx += 1
    cap.release()

    if not frame_buffer: return {}

    signals = {
        "fps": actual_fps, "timestamps": np.array(timestamps),
        "gaze_y": np.array(gaze_y_list), "gaze_x": np.array(gaze_x_list)
    }

    au_nums =["01", "02", "04", "06", "07", "12", "14", "17", "23", "24"]
    au_targets =[f"AU{au}" for au in au_nums]
    
    temp_dir = tempfile.mkdtemp()
    temp_paths =[]
    
    try:
        for i, img in enumerate(frame_buffer):
            tmp_p = os.path.join(temp_dir, f"f_{i:03d}.jpg")
            cv2.imwrite(tmp_p, img) 
            temp_paths.append(tmp_p)
        
        fex = detector.detect_image(temp_paths, batch_size=len(temp_paths))
        df = fex.to_dataframe() if hasattr(fex, "to_dataframe") else pd.DataFrame(fex)
        
        for au in au_nums:
            target_key = f"AU{au}"
            col = None
            for c in[f"AU{au}_r", f"AU{au}", f"AU{au}_c", f"AU{int(au)}_r", f"AU{int(au)}", f"AU{int(au)}_c"]:
                if c in df.columns:
                    col = c
                    break
            if col:
                signals[target_key] = df[col].astype(float).fillna(0.0).values
            else:
                signals[target_key] = np.zeros(len(frame_buffer))
                
    except Exception as e:
        print(f"⚠️ PyFeat Failed: {e}")
        for t in au_targets: signals[t] = np.zeros(len(frame_buffer))
    finally:
        if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
            
    min_len = min(len(signals["timestamps"]), len(signals["gaze_y"]))
    for k in signals.keys():
        if k != "fps" and isinstance(signals[k], np.ndarray):
            signals[k] = signals[k][:min_len]

    return signals

def z_score_symbolic_parser(signals: Dict) -> str:
    if not signals or "timestamps" not in signals: return "[Error] No signals."
    timestamps = signals["timestamps"]
    events =[]

    track_dict = {
        "gaze_y": {"name": "EYE-ROLL (Upward Gaze)", "type": "positive"},
        "gaze_x": {"name": "SIDE-EYE (Lateral Gaze)", "type": "absolute"},
        "AU14": {"name": "SMIRK_CONTEMPT (Unilateral Lip Tightener)", "type": "positive"},
        "AU24": {"name": "LIP_PRESS (Suppressed Emotion)", "type": "positive"},
        "AU23": {"name": "LIP_TIGHTEN (Frustration)", "type": "positive"},
        "AU07": {"name": "SQUINT (Suspicion/Disbelief)", "type": "positive"},
        "AU04": {"name": "BROW_FURROW (Disapproval)", "type": "positive"},
        "AU01": {"name": "BROW_RAISE (Surprise/Disbelief)", "type": "positive"},
    }

    for key, meta in track_dict.items():
        if key not in signals: continue
        arr = signals[key]
        if len(arr) == 0: continue
        
        p25, p75 = np.percentile(arr, 25), np.percentile(arr, 75)
        neutral_mask = (arr >= p25) & (arr <= p75)
        mu = np.mean(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.mean(arr)
        sigma = np.std(arr[neutral_mask]) if np.sum(neutral_mask) > 0 else np.std(arr)
        sigma = max(sigma, 0.05) 
        
        z_scores = (arr - mu) / sigma
        
        if meta["type"] == "positive": anomalies = z_scores > Z_SCORE_THRESH_MICRO
        else: anomalies = np.abs(z_scores) > Z_SCORE_THRESH_MICRO

        in_event = False
        start_idx = 0
        for i, is_abnormal in enumerate(anomalies):
            if is_abnormal and not in_event:
                in_event, start_idx = True, i
            elif not is_abnormal and in_event:
                in_event = False
                dur = timestamps[i-1] - timestamps[start_idx]
                if dur >= MIN_DURATION_SEC:
                    peak_z = min(np.max(np.abs(z_scores[start_idx:i])), 15.0)
                    events.append({
                        "t_start": timestamps[start_idx],
                        "t_end": timestamps[i-1],
                        "event": meta["name"],
                        "intensity": round(float(peak_z), 1)
                    })

    deadpan_cols =["AU12", "AU14", "AU23", "AU24", "AU04", "AU01", "AU02", "AU07"]
    deadpan_stack = np.vstack([signals[c] for c in deadpan_cols if c in signals])
    if deadpan_stack.size:
        mask_deadpan = np.all(deadpan_stack < 1.0, axis=0) 
        in_event, start_idx = False, 0
        for i, is_dp in enumerate(mask_deadpan):
            if is_dp and not in_event:
                in_event, start_idx = True, i
            elif not is_dp and in_event:
                in_event = False
                if timestamps[i-1] - timestamps[start_idx] >= 0.5:
                    events.append({"t_start": timestamps[start_idx], "t_end": timestamps[i-1], "event": "DEADPAN (Emotionless)", "intensity": 5.0})

    events = sorted(events, key=lambda x: x["t_start"])
    
    report = "[DETECTED VISUAL EVENTS (Z-Score Calibrated)]:\n"
    if not events: report += "- Normal / Neutral Gaze and Expressions.\n"
    else:
        for e in events: report += f"- At {e['t_start']:.1f}s - {e['t_end']:.1f}s: {e['event']} (Intensity: {e['intensity']})\n"
    return report

# =====================================================================
# Phase 2: ToM Reasoning
# =====================================================================
def phase2_tom_draft(symbolic_report: str) -> str:
#     prompt = f"""
# You are a behavioral psychology expert. Recent cognitive research indicates that human sarcasm detection is a holistic, non-sequential process rather than a step-by-step logical deduction.

# ### VISUAL EVIDENCE (Tensor of Cues):
# {symbolic_report}

# ### HOLISTIC INTEGRATION GUIDELINES:
# Do NOT think step-by-step. Instead, treat the visual evidence as a "Bag of Cues" and synthesize them instantly across three independent, parallel dimensions:
# - Dimension 1 (Eye Dynamics): Look for high-intensity Gaze anomalies (e.g., EYE-ROLL, SIDE-EYE).
# - Dimension 2 (Lower Face Dynamics): Look for suppressed or asymmetric mouth movements (e.g., SMIRK, LIP_PRESS, LIP_TIGHTEN, DEADPAN).
# - Dimension 3 (Upper Face Dynamics): Look for cognitive dissonance markers (e.g., SQUINT, BROW_FURROW, BROW_RAISE).[Sarcasm Synthesis Rule]: 
# Sarcasm is highly probable if there is a strong anomaly (high Z-score intensity) in ANY of these orthogonal dimensions, or a synergistic combination of them (e.g., an eye-roll coupled with a smirk). If the evidence shows "Normal / Neutral", it lacks the necessary leakage for sarcasm.

# ### OUTPUT FORMAT:
# Provide your response in this exact format:
# Reasoning:[Write 1-2 sentences holistically synthesizing the dimensions to state if sufficient sarcastic cues are present]
# Final_JSON: {{"sarcasm": true}} or {{"sarcasm": false}}
#     """


    prompt = f"""
You are an AI assistant specialized in facial expression analysis. Your task is to analyze the detected visual events and classify whether the person is showing sarcasm.

### VISUAL EVIDENCE:
{symbolic_report}

### INSTRUCTIONS:
Read the visual evidence and map them to the basic emotional states described above. Based on these emotional indicators, predict if the overall expression constitutes sarcasm. If the evidence primarily points to basic emotions like surprise, anger, or frustration, it may not necessarily be sarcasm.

### OUTPUT FORMAT:
Provide your response in this exact format:
Reasoning:[Write your analysis mapping the visual events to basic emotions and conclude if it is sarcasm]
Final_JSON: {{"sarcasm": true}} or {{"sarcasm": false}}
"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.1
        )
        content = response.choices[0].message.content.strip()
        if not content:
            return "Reasoning: Empty response. \nFinal_JSON: {\"sarcasm\": false}"
        return content
    except Exception as e:
        return f"Reasoning: Error {e}. \nFinal_JSON: " + "{\"sarcasm\": false}"


def clean_and_parse_json(content: str) -> Dict:
    content = str(content).strip()
    is_sarcastic = False
    if re.search(r'"sarcasm"\s*:\s*true', content, re.IGNORECASE):
        is_sarcastic = True
        
    reasoning = ""
    m = re.search(r'Reasoning:\s*(.*?)(?=Final_JSON|$)', content, re.DOTALL | re.IGNORECASE)
    if m: reasoning = m.group(1).strip()
        
    return {"Is_Sarcastic": is_sarcastic, "reasoning": reasoning}


# =====================================================================
# 主流程
# =====================================================================
def main():
    global face_mesh
    
    # 初始化输出目录
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    # 读取输入 JSON
    with open(INPUT_DATA_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 读取已处理的推断结果记录（用于断点续传）
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try: processed_ids.add(json.loads(line)['video_id'])
                except: pass
                
    # 👇 新增：加载本地特征解析缓存 (Cache)
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r', encoding='utf-8') as f:
            try:
                cache = json.load(f)
                print(f"📦 已加载本地视觉特征缓存: 包含 {len(cache)} 条记录")
            except json.JSONDecodeError:
                print("⚠️ 缓存文件读取失败，将重新建立。")
    
    # 梳理需要处理的待办列表
    items = list(data.items()) if isinstance(data, dict) else list(enumerate(data))
    items_to_process =[]
    need_visual_processing = False
    
    for vid_id, item in items:
        actual_vid_id = item.get('video_id', str(vid_id)) if isinstance(data, list) else str(vid_id)
        if actual_vid_id not in processed_ids:
            items_to_process.append((actual_vid_id, item))
            # 检查是否有视频既没处理过，又没有缓存，需要跑模型
            if actual_vid_id not in cache:
                need_visual_processing = True

    # 👇 新增：懒加载策略。如果全部都命中了缓存，直接跳过初始化 MediaPipe/PyFeat，极致提速
    detector = None
    if need_visual_processing:
        print("🚀 初始化 Perception 基建 (MediaPipe & PyFeat) ...")
        mp_face_mesh = mp.solutions.face_mesh
        face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)
        # 注意：需要确保你之前已经导入了 feat.Detector
        from feat import Detector
        detector = Detector(face_model="retinaface", landmark_model="mobilefacenet", au_model="svm", emotion_model="resmasknet", device="cuda")
    else:
        print("⚡ 所有待处理视频均已命中本地缓存，跳过加载庞大的视觉感知模型！")

    print(f"🚀 开始处理 Ablation Pipeline (w/o Phase 3: No Critic-Guided Reflection)")

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items_to_process, desc="Reasoning"):
            video_path = os.path.join(VIDEO_FOLDER, f"{vid_id}.mp4")

            try:
                # ================= 新增：缓存存取逻辑 =================
                if vid_id in cache:
                    symbolic_report = cache[vid_id]
                else:
                    if not os.path.exists(video_path): 
                        print(f"⚠️ 找不到视频文件: {video_path}")
                        continue
                        
                    # 1. 提取丰富的 AU 信号并解析
                    signals = extract_visual_signals_optimized(video_path, detector)
                    if not signals: continue 
                    
                    symbolic_report = z_score_symbolic_parser(signals)
                    
                    # 1.1 将结果存入 Cache 并立即写入磁盘
                    cache[vid_id] = symbolic_report
                    with open(CACHE_FILE, 'w', encoding='utf-8') as cf:
                        json.dump(cache, cf, ensure_ascii=False, indent=4)
                # ======================================================
                
                # 2. ToM 逻辑推理 (生成第一版草稿)
                current_draft = phase2_tom_draft(symbolic_report)
                
                # 3. 【消融】直接跳过 Critic 打分和修改，强行采纳第一版草稿
                final_json = clean_and_parse_json(current_draft)
                
                # ================= 结果构造与写入 =================
                result_entry = {
                    "video_id": vid_id,
                    "ground_truth": item.get("sarcasm", None),
                    "prediction": final_json.get("Is_Sarcastic", False),
                    "neuro_symbolic_trace": {
                        "symbolic_report": symbolic_report, 
                        "final_reasoning": final_json.get("reasoning", ""),
                        "critique_history": ["[ABLATION] Phase 3 removed. No self-reflection performed."],
                        "raw_output": current_draft
                    }
                }
                
                out_f.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
                out_f.flush()
                
            except Exception as e:
                print(f"❌ Error processing {vid_id}: {e}")
                err_entry = {"video_id": vid_id, "error": str(e)}
                out_f.write(json.dumps(err_entry) + "\n")
                out_f.flush()

    print("✅ 消融实验处理完毕。结果已保存至:", OUTPUT_FILE)

if __name__ == "__main__":
    main()

📦 已加载本地视觉特征缓存: 包含 690 条记录
⚡ 所有待处理视频均已命中本地缓存，跳过加载庞大的视觉感知模型！
🚀 开始处理 Ablation Pipeline (w/o Phase 3: No Critic-Guided Reflection)


Reasoning:   2%|▏         | 12/690 [01:35<1:21:08,  7.18s/it]

# 先GLM识别gaze，再7b VLM做sarcasm判断

In [14]:
import json
import os
import io
import argparse
import base64
import glob
from typing import List, Dict, Any
from tqdm import tqdm
import ollama 
from zai import ZhipuAiClient 
import cv2
from PIL import Image

# ================= 配置区域 =================
# API 配置
ZHIPU_API_KEY = "bb4d71f420554dbcb4751c1220b6d49e.LkKhpmPLO8lH0hdp" # ⚠️ 请注意保护你的API Key，不要上传到公开仓库
GLM_MODEL_NAME = "glm-4.6v" 
# GLM_MODEL_NAME = "glm-4.6v-Flash"

# 本地 LLM 配置
LLM_MODEL_NAME = "qwen2.5:7b"

VIDEO_FOLDER = "./utterances_final_revise"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/glm6v_eyeroll_qwen_inference_1.jsonl"

TARGET_FRAME_COUNT = 20
# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """
    读取视频文件，均匀抽取指定数量的帧，并转为字节流列表 (适配Ollama输入)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    step = max(1, total_frames // target_count)
    frame_bytes_list = []
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if count % step == 0 and saved_count < target_count:
            # OpenCV BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 缩放图片以节省显存和Token (推荐长边512或448)
            pil_img.thumbnail((720, 720)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=80)
            frame_bytes_list.append(img_byte_arr.getvalue())
            saved_count += 1
            
        count += 1

    cap.release()
    return frame_bytes_list

def stage1_extract_gaze_glm(video_path: str, speaker_name: str, client: ZhipuAiClient) -> str:
    """
    Stage 1: 使用 GLM-4v API 提取眼神描述
    直接发送 Base64 编码的视频文件
    """
    if not os.path.exists(video_path):
        return "Video file not found."

    try:
        # 1. 视频转 Base64
        with open(video_path, "rb") as video_file:
            video_base64 = base64.b64encode(video_file.read()).decode('utf-8')

        # 2. 构建 Prompt
        prompt =         f"""
        You are a video-based Gaze Analysis Expert.
        Task: Determine whether the speaker {speaker_name} shows (1) a contemptuous squint/side-eye or (2) a clear eye-roll at any time in the video.
        Procedure:
        1.	Scan the full video and list up to 3 candidate moments with timestamps (mm:ss–mm:ss).
        2.	For each moment, describe objective cues only. Evidence:[ type: “eye_roll”|“side_eye_squint”, start:“mm:ss”, end:“mm:ss”, cues:”…” ]
        3.	Decide: Squint/side-eye present? Eye-roll present? 
        Output format (no extra text):
        eye_roll: Yes/No
        side_eye_squint: Yes/No
        final: Yes if either is Yes, else No
    
        """

        # 3. 调用 API
        response = client.chat.completions.create(
            model=GLM_MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "video_url",
                            "video_url": {
                                "url": video_base64
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ],
            #  temperature=0.1 
        )
        
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"⚠️ GLM API Error on {video_path}: {e}")
        return "Error extracting gaze description."

def stage2_infer_sarcasm(utterance: str, gaze_desc: str, context: str, context_speakers: List[str]) -> dict:
    """
    Stage 2: 使用本地 LLM (Qwen2.5) 进行讽刺推理
    """
    
    prompt = (
        f"You are an expert in analyzing multimodal social interactions, specifically detecting sarcasm. "
        f"You will be provided with video frames from the TV show and the transcript of the conversation.\n\n"
        
        f"### Context (Previous Conversation):\n{context}\n"
        f"### Target Utterance:\n- {context_speakers}: \"{utterance}\"\n\n"
        f"Critical: ### Did the speaker show any signs of sarcasm? {gaze_desc}\n\n"
        
        f"### Task:\n"
        f"Analyze the speaker's facial expressions, body language, and the context of the conversation. "
        f"Determine if the target utterance is SARCASTIC or NOT SARCASTIC.\n\n"
        f"If any sign of sarcasm (eye_roll, side_eye_squint) is detected, it is SARCASTIC.\n\n"
        
        f"### Response Format:\n"
        f"1. Thinking out loud: Briefly analyze the visual cues and context (2-3 sentences).\n"
        f"2. Final Answer: Return strictly valid JSON format: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}."
    )

    try:
        response = ollama.chat(
            model=LLM_MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.1, 'num_ctx': 15000} 
        )
        content = response['message']['content']
        
        return parse_llm_json(content)
        
    except Exception as e:
        print(f"⚠️ LLM Error: {e}")
        return {"sarcasm": False, "error": str(e), "raw_response": ""}

def parse_llm_json(content: str) -> dict:
    """
    解析 LLM 可能带有 Markdown 格式的 JSON 输出
    """
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    try:
        if "```json" in content:
            json_str = content.split("```json")[1].split("```")[0].strip()
            return json.loads(json_str)
        elif "{" in content and "}" in content:
            start = content.find("{")
            end = content.rfind("}") + 1
            json_str = content[start:end]
            return json.loads(json_str)
    except Exception:
        pass
    
    return {"sarcasm": False, "parsing_error": True, "raw_response": content}

def main():
    args = parse_args()
    
    # 初始化智谱客户端
    zhipu_client = ZhipuAiClient(api_key=ZHIPU_API_KEY)

    # 确保输出目录存在
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入数据
    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 检查已处理的数据（断点续传）
    processed_ids = set()
    if os.path.exists(args.output_file):
        with open(args.output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    res = json.loads(line)
                    if 'video_id' in res: 
                        processed_ids.add(res['video_id'])
                except:
                    pass

    print(f"🚀 开始处理 Pipeline: VLM({GLM_MODEL_NAME}) -> LLM({LLM_MODEL_NAME})")
    
    items = data.items() if isinstance(data, dict) else enumerate(data)

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing Videos"):
            
            # ID 处理
            if isinstance(data, list):
                current_id = item.get('video_id', str(vid_id))
            else:
                current_id = vid_id
            
            if current_id in processed_ids:
                continue

            # 构建视频路径 (假设是 .mp4)
            video_file = f"{current_id}.mp4"
            video_path = os.path.join(args.video_folder, video_file)
            
            # 准备上下文数据
            speaker = item.get("speaker", "The Speaker")
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            # 格式化上下文
            context_str = ""
            for s, u in zip(context_speakers, context_list):
                context_str += f"- {s}: {u}\n"

            # ================= Stage 1: GLM-4v 提取 Gaze =================
            # 这里不再需要抽帧，直接传视频路径
            gaze_description = stage1_extract_gaze_glm(video_path, speaker, zhipu_client)
            # frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)
            
            # 如果GLM调用失败，gaze_description 会是错误信息，需要处理
            if "Error" in gaze_description:
                print(f"Skipping {current_id} due to VLM error.")
                continue

            # ================= Stage 2: LLM 推理 Sarcasm =================
            llm_result = stage2_infer_sarcasm(utterance, gaze_description, context_str, context_speakers)
            
            result = {
                "video_id": current_id,
                "ground_truth": item.get("sarcasm", None), 
                "gaze_description": gaze_description,      
                "prediction_reasoning": llm_result.get("raw_response", llm_result.get("Thinking out loud", "")),
                "sarcasm_prediction": llm_result.get("sarcasm", False)
            }

            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 处理完成，结果已保存至 {args.output_file}")

if __name__ == "__main__":
    main()

🚀 开始处理 Pipeline: VLM(glm-4.6v) -> LLM(qwen2.5:7b)


Processing Videos:   0%|          | 0/690 [00:00<?, ?it/s]

Processing Videos:  63%|██████▎   | 437/690 [45:33<43:56, 10.42s/it]   

⚠️ GLM API Error on ./utterances_final_revise/2_429.mp4: Error code: 400, with error text {"contentFilter":[{"level":2,"role":"user"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}
Skipping 2_429 due to VLM error.


Processing Videos: 100%|██████████| 690/690 [2:17:29<00:00, 11.96s/it]   

✅ 处理完成，结果已保存至 ./results/glm6v_eyeroll_qwen_inference_1.jsonl


# Mediapipe + VLM

In [12]:
import cv2
import mediapipe as mp
import numpy as np
import os
import json
import argparse
import ollama
from tqdm import tqdm
from typing import List, Dict, Any
from PIL import Image
import io
from feat import Detector
from au_semantic import detect_au_events_from_video, StateMachineConfig


# ================= 配置部分 =================
LLM_MODEL_NAME = "qwen2.5vl:7b"

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/cv_gaze_qwenvlm_noutterances_prompt.jsonl"

# --- 视觉阈值配置 (根据你的要求) ---
EYE_ROLL_THRESHOLD = 0.88   
SIDE_EYE_LEFT_THR = 0.32    
SIDE_EYE_RIGHT_THR = 0.68   
EAR_BLINK_THRESH = 0.15     
EAR_SQUINT_THRESH = 0.22    
TARGET_FRAME_COUNT = 20

# 时间平滑配置
CONSECUTIVE_FRAMES = 3 

# ================= MediaPipe 初始化 =================
print("正在初始化 MediaPipe...")
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=3, 
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """
    读取视频文件，均匀抽取指定数量的帧，并转为字节流列表 (适配Ollama输入)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    step = max(1, total_frames // target_count)
    frame_bytes_list = []
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if count % step == 0 and saved_count < target_count:
            # OpenCV BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 缩放图片以节省显存和Token (推荐长边512或448)
            pil_img.thumbnail((720, 720)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=80)
            frame_bytes_list.append(img_byte_arr.getvalue())
            saved_count += 1
            
        count += 1

    cap.release()
    return frame_bytes_list

# ================= 视觉分析函数 (Stage 1) =================

def calculate_ear(landmarks, top_idx, bottom_idx, inner_idx, outer_idx):
    """计算眼睛纵横比 (Eye Aspect Ratio)"""
    p_top = np.array([landmarks[top_idx].x, landmarks[top_idx].y])
    p_bottom = np.array([landmarks[bottom_idx].x, landmarks[bottom_idx].y])
    p_inner = np.array([landmarks[inner_idx].x, landmarks[inner_idx].y])
    p_outer = np.array([landmarks[outer_idx].x, landmarks[outer_idx].y])

    v_dist = np.linalg.norm(p_top - p_bottom)
    h_dist = np.linalg.norm(p_inner - p_outer)

    if h_dist == 0: return 0
    return v_dist / h_dist

def get_gaze_data(landmarks):
    """
    返回: (h_ratio, v_ratio, avg_ear, is_valid)
    """
    LEFT_IRIS = 468
    LEFT_TOP, LEFT_BOTTOM = 159, 145
    LEFT_INNER, LEFT_OUTER = 133, 33
    
    RIGHT_IRIS = 473
    RIGHT_TOP, RIGHT_BOTTOM = 386, 374
    RIGHT_INNER, RIGHT_OUTER = 362, 263

    def _get_ratio(iris_idx, inner_idx, outer_idx, top_idx, bottom_idx):
        p_iris = landmarks[iris_idx]
        p_top = landmarks[top_idx]
        p_bottom = landmarks[bottom_idx]
        p_inner = landmarks[inner_idx]
        p_outer = landmarks[outer_idx]

        # 安全检查：眼睛太小（如向下看或闭眼）时，计算Ratio会数值爆炸
        eye_height = abs(p_bottom.y - p_top.y)

        # 水平比率
        eye_left_x = min(p_inner.x, p_outer.x)
        eye_right_x = max(p_inner.x, p_outer.x)
        h_ratio = (p_iris.x - eye_left_x) / (eye_right_x - eye_left_x + 1e-6)

        # 垂直比率
        v_ratio = (p_bottom.y - p_iris.y) / (eye_height + 1e-6)
        
        return h_ratio, v_ratio

    # EAR 计算
    ear_left = calculate_ear(landmarks, LEFT_TOP, LEFT_BOTTOM, LEFT_INNER, LEFT_OUTER)
    ear_right = calculate_ear(landmarks, RIGHT_TOP, RIGHT_BOTTOM, RIGHT_INNER, RIGHT_OUTER)
    avg_ear = (ear_left + ear_right) / 2.0

    # 视线比率计算
    h_left, v_left = _get_ratio(LEFT_IRIS, LEFT_INNER, LEFT_OUTER, LEFT_TOP, LEFT_BOTTOM)
    h_right, v_right = _get_ratio(RIGHT_IRIS, RIGHT_INNER, RIGHT_OUTER, RIGHT_TOP, RIGHT_BOTTOM)
    
    # 如果任意一只眼睛数据无效（如高度不够），则认为本帧不可用
    if h_left is None or h_right is None:
        return 0, 0, avg_ear, False

    avg_h = (h_left + h_right) / 2.0
    avg_v = (v_left + v_right) / 2.0

    return avg_h, avg_v, avg_ear, True

def detect_gaze_behavior(file_path):
    """
    替代原有的 VLM get_gaze 函数。
    返回: (detected: bool, description: str)
    """
    cap = cv2.VideoCapture(file_path)
    if not cap.isOpened():
        return False, "Error: Could not open video"

    consecutive_counter = 0
    last_behavior = None
    detected_final = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)
        
        frame_has_behavior = False
        current_frame_behavior = None

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                h_ratio, v_ratio, ear, is_valid = get_gaze_data(face_landmarks.landmark)
                
                # 1. 基础过滤
                if ear < EAR_BLINK_THRESH or not is_valid:
                    continue 

                # 2. 判定逻辑
                behavior = None
                
                # B. 斜眼
                if h_ratio < SIDE_EYE_LEFT_THR or h_ratio > SIDE_EYE_RIGHT_THR:
                    behavior = "SIDE-EYE (Glancing Sideways)"

                # A. 翻白眼 (优先级高)
                elif v_ratio > EYE_ROLL_THRESHOLD:
                    behavior = "EYE-ROLL (Rolling Eyes Upwards)"

                if behavior:
                    frame_has_behavior = True
                    current_frame_behavior = behavior
                    break # 这一帧有人物触发即可
        
        # --- 连续帧确认逻辑 ---
        if frame_has_behavior:
            if current_frame_behavior == last_behavior:
                consecutive_counter += 1
            else:
                consecutive_counter = 1
                last_behavior = current_frame_behavior
            
            if consecutive_counter >= CONSECUTIVE_FRAMES:
                detected_final = last_behavior
                break # Early Exit: 只要确认检测到，立即返回
        else:
            consecutive_counter = 0
            last_behavior = None

    cap.release()
    
    if detected_final:
        return True, f"Detected strong visual signal: {detected_final}."
    else:
        return False, "No specific sarcastic eye movements detected (Normal Gaze)."

# ================= LLM 推理函数 (Stage 2) =================

def stage2_infer_sarcasm(frames: List[bytes], utterance: str, gaze_desc: str, context: str, context_speakers: List[str]) -> dict:
    """
    使用本地 LLM (Qwen2.5) 结合 Gaze 结果进行推理
    """
    prompt = (
        # f"You are a social psychology expert. Your task is to detect sarcasm by analyzing the conflict between a speaker's visual behavior (Gaze) and their spoken words (Text).\n\n"
        f"You are a social psychology expert. Your task is to detect sarcasm by analyzing the speaker's visual behaviors.\n\n"
        
        f"### CASE DATA:\n"
        # f"1. PREVIOUS CONTEXT:\n{context}\n"
        # f"2. SPEAKER & UTTERANCE (Text):\n- {context_speakers}: \"{utterance}\"\n"
        f"1. VISUAL EVIDENCE (Gaze):\n- {gaze_desc}\n\n"
        
        f"### CAUSAL REASONING STEPS:\n"
        f"Perform the analysis strictly following this logic chain:\n"
        f"STEP 1 [Visual Intent]: Analyze the '{gaze_desc}'. Does this gaze gesture (e.g., eye-roll, looking away) typically convey negative emotion (annoyance, disbelief, boredom)?\n"
        f"Focus primarily on gaze (e.g., eye-roll, side-eye), and use facial events (e.g., smirk, lip press, squint, brow raise/furrow, deadpan) as supporting cues to infer the speaker's attitude (annoyance, disbelief, contempt, boredom).\n"
        f"NOTE: If gaze clearly indicates an attitude, do not override it with weak AU evidence; AU is only for disambiguation when gaze could be non-sarcastic.\n"
        # f"STEP 2 [Text Intent]: Analyze the literal meaning of \"{utterance}\". Is the sentiment positive, neutral, or negative?\n"
        # f"STEP 3 [Conflict Check]: Compare STEP 1 and STEP 2. Is there a contradiction?\n"
        # f"   - IF YES (Contradiction) -> The speaker is being SARCASTIC.\n"
        # f"   - IF NO (Consistent) -> The speaker is NOT SARCASTIC.\n\n"
        
        f"### OUTPUT FORMAT:\n"
        f"Provide your response in this exact format:\n"
        f"Reasoning: [Your step-by-step analysis here based on the logic chain]\n"
        f"Final_JSON: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}"
    )


    try:
        response = ollama.chat(
            model=LLM_MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt, 'images': frames}],
            options={'temperature': 0.1, 'num_ctx': 8000} 
        )
        content = response['message']['content']
        return parse_llm_json(content)
        
    except Exception as e:
        print(f"⚠️ LLM Error: {e}")
        return {"sarcasm": False, "error": str(e), "raw_response": ""}

def parse_llm_json(content: str) -> dict:
    """解析 LLM 输出的 JSON"""
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    try:
        # 尝试提取 Markdown 代码块中的 JSON
        if "```json" in content:
            json_str = content.split("```json")[1].split("```")[0].strip()
            return json.loads(json_str)
        elif "{" in content and "}" in content:
            start = content.find("{")
            end = content.rfind("}") + 1
            json_str = content[start:end]
            return json.loads(json_str)
    except Exception:
        pass
    
    return {"sarcasm": False, "parsing_error": True, "raw_response": content}

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

# ================= 主流程 =================

def main():
    args = parse_args()

    # 确保输出目录存在
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入数据
    if not os.path.exists(args.input_file):
        print(f"❌ Input file not found: {args.input_file}")
        return

    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 检查已处理的数据（断点续传）
    processed_ids = set()
    if os.path.exists(args.output_file):
        with open(args.output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    res = json.loads(line)
                    if 'video_id' in res: 
                        processed_ids.add(res['video_id'])
                except:
                    pass

    print(f"🚀 开始处理 Pipeline: CV(MediaPipe) -> LLM({LLM_MODEL_NAME})")
    print(f"Thresholds -> EyeRoll: {EYE_ROLL_THRESHOLD}, SideEye: {SIDE_EYE_LEFT_THR}/{SIDE_EYE_RIGHT_THR}")
    
    items = data.items() if isinstance(data, dict) else enumerate(data)

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing Videos"):
            
            # ID 处理
            if isinstance(data, list):
                current_id = item.get('video_id', str(vid_id))
            else:
                current_id = vid_id
            
            if current_id in processed_ids:
                continue

            # 构建视频路径
            video_file = f"{current_id}.mp4"
            video_path = os.path.join(args.video_folder, video_file)
            
            if not os.path.exists(video_path):
                # 尝试加上 .mp4 后缀查找，或者路径不对跳过
                pass 
            
            # 准备上下文数据
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            context_str = ""
            for s, u in zip(context_speakers, context_list):
                context_str += f"- {s}: {u}\n"

            # ================= Stage 1: CV 检测 Gaze =================
            # 直接传入视频路径，返回检测结果描述
            is_detected, gaze_description = detect_gaze_behavior(video_path)
            
            # ================= Stage 2: LLM 推理 Sarcasm =================

            frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)
            llm_result = stage2_infer_sarcasm(frames, utterance, gaze_description, context_str, context_speakers)
            
            result = {
                "video_id": current_id,
                "ground_truth": item.get("sarcasm", None), 
                "visual_detected": is_detected,
                "gaze_description": gaze_description,      
                "prediction_reasoning": llm_result.get("raw_response", llm_result.get("Thinking out loud", "")),
                "sarcasm_prediction": llm_result.get("sarcasm", False)
            }

            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 处理完成，结果已保存至 {args.output_file}")

if __name__ == "__main__":
    main()

正在初始化 MediaPipe...
🚀 开始处理 Pipeline: CV(MediaPipe) -> LLM(qwen2.5vl:7b)
Thresholds -> EyeRoll: 0.88, SideEye: 0.32/0.68


Processing Videos:  30%|███       | 207/690 [21:28<1:09:49,  8.67s/it]

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


Processing Videos:  49%|████▉     | 339/690 [37:43<2:24:03, 24.62s/it]

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


Processing Videos:  87%|████████▋ | 602/690 [1:05:12<10:37,  7.24s/it]

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


Processing Videos: 100%|██████████| 690/690 [1:13:12<00:00,  6.37s/it]

✅ 处理完成，结果已保存至 ./results/cv_gaze_qwenvlm_noutterances_prompt.jsonl


# mediapipe + AU + VLM

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import json
import argparse
import ollama
from tqdm import tqdm
from typing import List, Dict, Any
from PIL import Image
import io
from feat import Detector
from au_semantic import detect_au_events_from_video, StateMachineConfig


# ================= 配置部分 =================
LLM_MODEL_NAME = "qwen2.5vl:7b"

VIDEO_FOLDER = "./utterances_eyeroll"  
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/cv_au_qwenvl_new.jsonl"

# --- 视觉阈值配置 (根据你的要求) ---
EYE_ROLL_THRESHOLD = 0.88   
SIDE_EYE_LEFT_THR = 0.32    
SIDE_EYE_RIGHT_THR = 0.68   
EAR_BLINK_THRESH = 0.15     
EAR_SQUINT_THRESH = 0.22    
TARGET_FRAME_COUNT = 20

# 时间平滑配置
CONSECUTIVE_FRAMES = 3 

# ================= MediaPipe 初始化 =================
print("正在初始化 MediaPipe...")
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=3, 
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """
    读取视频文件，均匀抽取指定数量的帧，并转为字节流列表 (适配Ollama输入)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    step = max(1, total_frames // target_count)
    frame_bytes_list = []
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if count % step == 0 and saved_count < target_count:
            # OpenCV BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 缩放图片以节省显存和Token (推荐长边512或448)
            pil_img.thumbnail((720, 720)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=80)
            frame_bytes_list.append(img_byte_arr.getvalue())
            saved_count += 1
            
        count += 1

    cap.release()
    return frame_bytes_list

# ================= 视觉分析函数 (Stage 1) =================

def calculate_ear(landmarks, top_idx, bottom_idx, inner_idx, outer_idx):
    """计算眼睛纵横比 (Eye Aspect Ratio)"""
    p_top = np.array([landmarks[top_idx].x, landmarks[top_idx].y])
    p_bottom = np.array([landmarks[bottom_idx].x, landmarks[bottom_idx].y])
    p_inner = np.array([landmarks[inner_idx].x, landmarks[inner_idx].y])
    p_outer = np.array([landmarks[outer_idx].x, landmarks[outer_idx].y])

    v_dist = np.linalg.norm(p_top - p_bottom)
    h_dist = np.linalg.norm(p_inner - p_outer)

    if h_dist == 0: return 0
    return v_dist / h_dist

def get_gaze_data(landmarks):
    """
    返回: (h_ratio, v_ratio, avg_ear, is_valid)
    """
    LEFT_IRIS = 468
    LEFT_TOP, LEFT_BOTTOM = 159, 145
    LEFT_INNER, LEFT_OUTER = 133, 33
    
    RIGHT_IRIS = 473
    RIGHT_TOP, RIGHT_BOTTOM = 386, 374
    RIGHT_INNER, RIGHT_OUTER = 362, 263

    def _get_ratio(iris_idx, inner_idx, outer_idx, top_idx, bottom_idx):
        p_iris = landmarks[iris_idx]
        p_top = landmarks[top_idx]
        p_bottom = landmarks[bottom_idx]
        p_inner = landmarks[inner_idx]
        p_outer = landmarks[outer_idx]

        # 安全检查：眼睛太小（如向下看或闭眼）时，计算Ratio会数值爆炸
        eye_height = abs(p_bottom.y - p_top.y)

        # 水平比率
        eye_left_x = min(p_inner.x, p_outer.x)
        eye_right_x = max(p_inner.x, p_outer.x)
        h_ratio = (p_iris.x - eye_left_x) / (eye_right_x - eye_left_x + 1e-6)

        # 垂直比率
        v_ratio = (p_bottom.y - p_iris.y) / (eye_height + 1e-6)
        
        return h_ratio, v_ratio

    # EAR 计算
    ear_left = calculate_ear(landmarks, LEFT_TOP, LEFT_BOTTOM, LEFT_INNER, LEFT_OUTER)
    ear_right = calculate_ear(landmarks, RIGHT_TOP, RIGHT_BOTTOM, RIGHT_INNER, RIGHT_OUTER)
    avg_ear = (ear_left + ear_right) / 2.0

    # 视线比率计算
    h_left, v_left = _get_ratio(LEFT_IRIS, LEFT_INNER, LEFT_OUTER, LEFT_TOP, LEFT_BOTTOM)
    h_right, v_right = _get_ratio(RIGHT_IRIS, RIGHT_INNER, RIGHT_OUTER, RIGHT_TOP, RIGHT_BOTTOM)
    
    # 如果任意一只眼睛数据无效（如高度不够），则认为本帧不可用
    if h_left is None or h_right is None:
        return 0, 0, avg_ear, False

    avg_h = (h_left + h_right) / 2.0
    avg_v = (v_left + v_right) / 2.0

    return avg_h, avg_v, avg_ear, True

def detect_gaze_behavior(file_path):
    """
    替代原有的 VLM get_gaze 函数。
    返回: (detected: bool, description: str)
    """
    cap = cv2.VideoCapture(file_path)
    if not cap.isOpened():
        return False, "Error: Could not open video"

    consecutive_counter = 0
    last_behavior = None
    detected_final = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)
        
        frame_has_behavior = False
        current_frame_behavior = None

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                h_ratio, v_ratio, ear, is_valid = get_gaze_data(face_landmarks.landmark)
                
                # 1. 基础过滤
                if ear < EAR_BLINK_THRESH or not is_valid:
                    continue 

                # 2. 判定逻辑
                behavior = None
                
                # B. 斜眼
                if h_ratio < SIDE_EYE_LEFT_THR or h_ratio > SIDE_EYE_RIGHT_THR:
                    behavior = "SIDE-EYE (Glancing Sideways)"

                # A. 翻白眼 (优先级高)
                elif v_ratio > EYE_ROLL_THRESHOLD:
                    behavior = "EYE-ROLL (Rolling Eyes Upwards)"

                if behavior:
                    frame_has_behavior = True
                    current_frame_behavior = behavior
                    break # 这一帧有人物触发即可
        
        # --- 连续帧确认逻辑 ---
        if frame_has_behavior:
            if current_frame_behavior == last_behavior:
                consecutive_counter += 1
            else:
                consecutive_counter = 1
                last_behavior = current_frame_behavior
            
            if consecutive_counter >= CONSECUTIVE_FRAMES:
                detected_final = last_behavior
                break # Early Exit: 只要确认检测到，立即返回
        else:
            consecutive_counter = 0
            last_behavior = None

    cap.release()
    
    if detected_final:
        return True, f"Detected strong visual signal: {detected_final}."
    else:
        return False, "No specific sarcastic eye movements detected (Normal Gaze)."

# ================= LLM 推理函数 (Stage 2) =================

def stage2_infer_sarcasm(frames: List[bytes], utterance: str, gaze_desc: str, context: str, context_speakers: List[str]) -> dict:
    """
    使用本地 LLM (Qwen2.5) 结合 Gaze 结果进行推理
    """
    # prompt = (
    #     f"You are a social psychology expert. Your task is to detect sarcasm by analyzing the conflict between a speaker's visual behavior (Gaze) and their spoken words (Text).\n\n"
        
    #     f"### CASE DATA:\n"
    #     f"1. PREVIOUS CONTEXT:\n{context}\n"
    #     f"2. SPEAKER & UTTERANCE (Text):\n- {context_speakers}: \"{utterance}\"\n"
    #     f"3. VISUAL EVIDENCE (Gaze):\n- {gaze_desc}\n\n"
        
    #     f"### CAUSAL REASONING STEPS:\n"
    #     f"Perform the analysis strictly following this logic chain:\n"
    #     f"STEP 1 [Visual Intent]: Analyze the '{gaze_desc}'. Does this gaze gesture (e.g., eye-roll, looking away) typically convey negative emotion (annoyance, disbelief, boredom)?\n"
    #     f"STEP 2 [Text Intent]: Analyze the literal meaning of \"{utterance}\". Is the sentiment positive, neutral, or negative?\n"
    #     f"STEP 3 [Conflict Check]: Compare STEP 1 and STEP 2. Is there a contradiction (e.g., Negative Gaze + Positive Text)?\n"
    #     f"   - IF YES (Contradiction) -> The speaker is being SARCASTIC.\n"
    #     f"   - IF NO (Consistent) -> The speaker is NOT SARCASTIC.\n\n"
        
    #     f"### OUTPUT FORMAT:\n"
    #     f"Provide your response in this exact format:\n"
    #     f"Reasoning: [Your step-by-step analysis here based on the logic chain]\n"
    #     f"Final_JSON: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}"
    # )

# 加了Au之后
    prompt = (
        f"You are a social psychology expert. Your task is to detect sarcasm by analyzing the conflict between a speaker's visual behavior (Visual Evidence) and their spoken words (Text).\n\n"
        
        f"### CASE DATA:\n"
        f"1. PREVIOUS CONTEXT:\n{context}\n"
        f"2. SPEAKER & UTTERANCE (Text):\n- {context_speakers}: \"{utterance}\"\n"
        f" VISUAL EVIDENCE (Gaze + Facial Expression AUs):\n- {gaze_desc}\n\n"
        
        f"### CAUSAL REASONING STEPS:\n"
        f"Perform the analysis strictly following this logic chain:\n"
        f"STEP 1 [Visual Intent]: Analyze the visual evidence. Focus primarily on gaze (e.g., eye-roll, side-eye), and use facial AU events (e.g., smirk, lip press, squint, brow raise/furrow, deadpan) as supporting cues to infer the speaker's attitude (annoyance, disbelief, contempt, boredom).\n"
        f"NOTE: If gaze clearly indicates an attitude, do not override it with weak AU evidence; AU is only for disambiguation when gaze could be non-sarcastic.\n"
        f"STEP 2 [Text Intent]: Analyze the literal meaning of \"{utterance}\". Is the sentiment positive, neutral, or negative?\n"
        f"STEP 3 [Conflict Check]: Compare STEP 1 and STEP 2. Is there a contradiction (e.g., Negative Visual Attitude + Positive Text)?\n"
        f"   - IF YES (Contradiction) -> The speaker is being SARCASTIC.\n"
        f"   - IF NO (Consistent) -> The speaker is NOT SARCASTIC.\n\n"
        
        f"### OUTPUT FORMAT:\n"
        f"Provide your response in this exact format:\n"
        f"Reasoning: [Your step-by-step analysis here based on the logic chain]\n"
        f"Final_JSON: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}"
    )

    try:
        response = ollama.chat(
            model=LLM_MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt, 'images': frames}],
            options={'temperature': 0.1, 'num_ctx': 8000} 
        )
        content = response['message']['content']
        return parse_llm_json(content)
        
    except Exception as e:
        print(f"⚠️ LLM Error: {e}")
        return {"sarcasm": False, "error": str(e), "raw_response": ""}

def parse_llm_json(content: str) -> dict:
    """解析 LLM 输出的 JSON"""
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    try:
        # 尝试提取 Markdown 代码块中的 JSON
        if "```json" in content:
            json_str = content.split("```json")[1].split("```")[0].strip()
            return json.loads(json_str)
        elif "{" in content and "}" in content:
            start = content.find("{")
            end = content.rfind("}") + 1
            json_str = content[start:end]
            return json.loads(json_str)
    except Exception:
        pass
    
    return {"sarcasm": False, "parsing_error": True, "raw_response": content}

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

# ================= 主流程 =================

def main():
    args = parse_args()

    # 确保输出目录存在
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入数据
    if not os.path.exists(args.input_file):
        print(f"❌ Input file not found: {args.input_file}")
        return

    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 检查已处理的数据（断点续传）
    processed_ids = set()
    if os.path.exists(args.output_file):
        with open(args.output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    res = json.loads(line)
                    if 'video_id' in res: 
                        processed_ids.add(res['video_id'])
                except:
                    pass

    print(f"🚀 开始处理 Pipeline: CV(MediaPipe) -> LLM({LLM_MODEL_NAME})")
    print(f"Thresholds -> EyeRoll: {EYE_ROLL_THRESHOLD}, SideEye: {SIDE_EYE_LEFT_THR}/{SIDE_EYE_RIGHT_THR}")
    
    items = data.items() if isinstance(data, dict) else enumerate(data)

    au_detector = Detector(
        face_model="retinaface",
        landmark_model="mobilefacenet",
        au_model="svm",
        emotion_model='svm'
        # device="cuda"
                )

    au_cfg = StateMachineConfig(
        onset_min_frames=5,
        offset_min_frames=3,
        min_dur_ms=250,
        max_dur_ms=2000,
        merge_gap_ms=150
    )

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing Videos"):
            
            # ID 处理
            if isinstance(data, list):
                current_id = item.get('video_id', str(vid_id))
            else:
                current_id = vid_id
            
            if current_id in processed_ids:
                continue

            # 构建视频路径
            video_file = f"{current_id}.mp4"
            video_path = os.path.join(args.video_folder, video_file)
            
            if not os.path.exists(video_path):
                # 尝试加上 .mp4 后缀查找，或者路径不对跳过
                pass 
            
            # 准备上下文数据
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            context_str = ""
            for s, u in zip(context_speakers, context_list):
                context_str += f"- {s}: {u}\n"

            # ================= Stage 1: CV 检测 Gaze =================
            # 直接传入视频路径，返回检测结果描述
            is_detected, gaze_description = detect_gaze_behavior(video_path)
            
            try:
                au_sem = detect_au_events_from_video(
                    video_path=video_path,
                    detector=au_detector,
                    cfg=au_cfg,
                    actor=context_speakers  # 你如果有speaker_id，可替换
                )
                au_events = au_sem["events"]
            except Exception as e:
                au_events = []
                au_sem = {"error": str(e), "events": []}
            
            # ================= Stage 2: LLM 推理 Sarcasm =================

            au_events_json = json.dumps(au_events, ensure_ascii=False)
            visual_semantics = (
                f"[Gaze semantics]\n{gaze_description}\n\n"
                f"[AU semantics JSON]\n{au_events_json}"
            )

            frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)
            # llm_result = stage2_infer_sarcasm(frames, utterance, gaze_description, context_str, context_speakers)
            llm_result = stage2_infer_sarcasm(frames, utterance, visual_semantics, context_str, context_speakers)
            
            result = {
                "video_id": current_id,
                "ground_truth": item.get("sarcasm", None), 
                "visual_detected": is_detected,
                "gaze_description": gaze_description,      
                "au_events": au_events,
                "prediction_reasoning": llm_result.get("raw_response", llm_result.get("Thinking out loud", "")),
                "sarcasm_prediction": llm_result.get("sarcasm", False)
            }

            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 处理完成，结果已保存至 {args.output_file}")

if __name__ == "__main__":
    main()

正在初始化 MediaPipe...
🚀 开始处理 Pipeline: CV(MediaPipe) -> LLM(qwen2.5vl:7b)
Thresholds -> EyeRoll: 0.88, SideEye: 0.32/0.68


100%|██████████| 36/36 [00:27<00:00,  1.33it/s]
/conda/kzh/envs/VIBE2/lib/python3.10/site-packages/feat/detector.py:981: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_output = pd.concat(batch_output)
Processing Videos:  59%|█████▊    | 404/690 [03:05<04:19,  1.10it/s]

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


Processing Videos:  59%|█████▉    | 408/690 [06:34<22:20,  4.75s/it]

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


...

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


100%|██████████| 266/266 [03:37<00:00,  1.22it/s]
/conda/kzh/envs/VIBE2/lib/python3.10/site-packages/feat/detector.py:981: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_output = pd.concat(batch_output)
100%|██████████| 372/372 [04:53<00:00,  1.27it/s]
/conda/kzh/envs/VIBE2/lib/python3.10/site-packages/feat/detector.py:981: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_output = pd.concat(batch_output)
100%|██████████| 220/220 [02:56<00:00,  1.25it/s]
/conda/kzh/envs/VIBE2/lib/python3.

⚠️ LLM Error: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


 54%|█████▍    | 76/140 [01:14<01:03,  1.01it/s]
/conda/kzh/envs/VIBE2/lib/python3.10/site-packages/feat/detector.py:981: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_output = pd.concat(batch_output)
100%|██████████| 140/140 [01:54<00:00,  1.23it/s]
/conda/kzh/envs/VIBE2/lib/python3.10/site-packages/feat/detector.py:981: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_output = pd.concat(batch_output)
100%|██████████| 171/171 [02:18<00:00,  1.24it/s]
/conda/kzh/envs/VIBE2/lib/python3.1

✅ 处理完成，结果已保存至 ./results/cv_au_qwenvl.jsonl


# 结果判断

In [4]:
import os
import json

INPUT_FILE = "./results/ablation_no_phase3_gemini.jsonl" 
# INPUT_FILE = "./results_ablation/remove_phase1_gaze.jsonl"  


def evaluate_and_extract_errors():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    # 初始化计数器
    stats = {
        "total": 0,
        "correct": 0,
        "tp": 0, # 真阳性 (GT=True, Pred=True)
        "tn": 0, # 真阴性 (GT=False, Pred=False)
        "fp": 0, # 假阳性 (GT=False, Pred=True) -> 误报
        "fn": 0  # 假阴性 (GT=True, Pred=False) -> 漏报
    }
    
    error_cases = []

    print(f"📂 正在读取并修复文件格式: {INPUT_FILE} ...")

    # 1. 读取整个文件内容到内存
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        content = f.read()

    # 2. 使用 raw_decode 逐个解析 JSON 对象
    decoder = json.JSONDecoder()
    pos = 0
    total_len = len(content)

    while pos < total_len:
        # 跳过空白字符 (空格、换行等)
        while pos < total_len and content[pos].isspace():
            pos += 1
        
        if pos >= total_len:
            break

        try:
            # raw_decode 返回 (解析出的对象, 解析结束后的新位置)
            item, end_pos = decoder.raw_decode(content, idx=pos)
            pos = end_pos # 更新位置指针
            
            # ================= 评估逻辑开始 =================
            gt = item.get("ground_truth")
            pred = item.get("prediction")

            # 跳过没有标签的数据 或 预测为空的数据
            if gt is None or pred is None:
                continue

            stats["total"] += 1

            # 确保 pred 是布尔值 (防止模型输出 "true"/"false" 字符串)
            if isinstance(pred, str):
                pred = True if pred.lower() == 'true' else False

            if gt == pred:
                stats["correct"] += 1
                if gt is True: stats["tp"] += 1
                else: stats["tn"] += 1
            else:
                # 记录错误类型
                if pred is True: 
                    stats["fp"] += 1
                    item["error_type"] = "False Positive (误报)"
                else: 
                    stats["fn"] += 1
                    item["error_type"] = "False Negative (漏报)"
                
                error_cases.append(item)
            # ================= 评估逻辑结束 =================

        except json.JSONDecodeError:
            # 如果解析失败，说明这里有非法字符，跳过一个字符尝试继续
            # print(f"⚠️ 解析警告: 在位置 {pos} 跳过非法字符")
            pos += 1

    # --- 计算指标 ---
    if stats["total"] == 0:
        print("❌ 未找到有效数据，请检查 JSON 字段名是否匹配 (ground_truth / sarcasm_prediction)")
        return

    acc = stats["correct"] / stats["total"]
    # 防止除以零
    precision = stats["tp"] / (stats["tp"] + stats["fp"]) if (stats["tp"] + stats["fp"]) > 0 else 0
    recall = stats["tp"] / (stats["tp"] + stats["fn"]) if (stats["tp"] + stats["fn"]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- 打印报告 ---
    print(f"\n{'='*30}")
    print(f"📊 评估结果 (样本数: {stats['total']})")
    print(f"{'='*30}")
    print(f"✅ Accuracy (准确率):  {acc:.2%}")
    print(f"🎯 Precision (查准率): {precision:.2%}")
    print(f"🔎 Recall    (查全率): {recall:.2%}")
    print(f"⚖️  F1-Score:         {f1:.2%}")
    print(f"{'-'*30}")
    print(f"详细统计:")
    print(f"  TP (讽刺且预测对): {stats['tp']}")
    print(f"  TN (正常且预测对): {stats['tn']}")
    print(f"  FP (误判为讽刺):   {stats['fp']} (模型太敏感)")
    print(f"  FN (漏判讽刺):     {stats['fn']} (模型没识别出)")
    print(f"{'='*30}")

if __name__ == "__main__":
    evaluate_and_extract_errors()

📂 正在读取并修复文件格式: ./results/ablation_no_phase3_gemini.jsonl ...

📊 评估结果 (样本数: 690)
✅ Accuracy (准确率):  71.45%
🎯 Precision (查准率): 72.84%
🔎 Recall    (查全率): 68.41%
⚖️  F1-Score:         70.55%
------------------------------
详细统计:
  TP (讽刺且预测对): 236
  TN (正常且预测对): 257
  FP (误判为讽刺):   88 (模型太敏感)
  FN (漏判讽刺):     109 (模型没识别出)


# new pipeline

In [ ]:
import json
import os

# ================= 配置 =================
# INPUT_FILE = "./results/vo_tom_sarcasm_vlm3.jsonl"  
INPUT_FILE = "./results-ablation/remove_phase1_gaze.jsonl"  

# =======================================

def evaluate_and_extract_errors():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    # 初始化计数器
    stats = {
        "total": 0,
        "correct": 0,
        "tp": 0, # GT=True, Pred=True
        "tn": 0, # GT=False, Pred=False
        "fp": 0, # GT=False, Pred=True (误报)
        "fn": 0  # GT=True, Pred=False (漏报)
    }
    
    error_cases = []

    print(f"正在读取文件: {INPUT_FILE} ...")

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line: 
                continue
            
            try:
                # 尝试解析 JSON
                item = json.loads(line)
                
                # === 关键修改点 ===
                # 根据你提供的数据，键名应该是 "prediction" 而不是 "sarcasm_prediction"
                gt = item.get("ground_truth")
                pred = item.get("prediction") 

                # 检查数据完整性
                if gt is None or pred is None:
                    # 如果某一行缺少 ground_truth 或 prediction，跳过并提示
                    # print(f"警告: 第 {line_num} 行数据缺失必要字段 (gt={gt}, pred={pred})")
                    continue

                stats["total"] += 1

                # 核心统计逻辑
                if gt == pred:
                    stats["correct"] += 1
                    if gt is True:
                        stats["tp"] += 1
                    else:
                        stats["tn"] += 1
                else:
                    # 记录错误
                    error_info = item.copy() # 复制一份，避免修改原数据影响后续
                    if pred is True: 
                        stats["fp"] += 1
                        error_info["error_type"] = "False Positive (误报: 实际没讽刺，模型说有)"
                    else: 
                        stats["fn"] += 1
                        error_info["error_type"] = "False Negative (漏报: 实际有讽刺，模型没看出来)"
                    
                    error_cases.append(error_info)

            except json.JSONDecodeError:
                print(f"❌ JSON 解析错误: 第 {line_num} 行")
                continue

    # --- 计算指标 ---
    if stats["total"] == 0:
        print("⚠️ 未找到有效数据。请检查 json 文件内容和键名是否匹配。")
        return

    acc = stats["correct"] / stats["total"]
    # 分母加极小值 1e-9 防止除以零报错
    precision = stats["tp"] / (stats["tp"] + stats["fp"] + 1e-9)
    recall = stats["tp"] / (stats["tp"] + stats["fn"] + 1e-9)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

    # --- 打印报告 ---
    print(f"\n{'='*40}")
    print(f"📊 评估结果 (Valid Samples: {stats['total']})")
    print(f"{'='*40}")
    print(f"✅ Accuracy (准确率):  {acc:.2%}")
    print(f"🎯 Precision (查准率): {precision:.2%}")
    print(f"🔎 Recall    (查全率): {recall:.2%}")
    print(f"⚖️  F1-Score:         {f1:.2%}")
    print(f"{'-'*40}")
    print(f"详细混淆矩阵:")
    print(f"  [TP] 讽刺且预测对: {stats['tp']}")
    print(f"  [TN] 正常且预测对: {stats['tn']}")
    print(f"  [FP] 误判为讽刺  : {stats['fp']} (模型太敏感)")
    print(f"  [FN] 漏判讽刺    : {stats['fn']} (模型没识别出)")
    print(f"{'='*40}")

    # --- 保存错误文件 (可选) ---
    # if error_cases:
    #     # 确保目录存在
    #     os.makedirs(os.path.dirname(ERROR_FILE) if os.path.dirname(ERROR_FILE) else '.', exist_ok=True)
    #     with open(ERROR_FILE, 'w', encoding='utf-8') as f:
    #         json.dump(error_cases, f, indent=4, ensure_ascii=False)
    #     print(f"\n📝 已将 {len(error_cases)} 个错误案例保存至: {ERROR_FILE}")

if __name__ == "__main__":
    evaluate_and_extract_errors()

正在读取文件: ./results/vo_tom_sarcasm_vlm3.jsonl ...

📊 评估结果 (Valid Samples: 690)
✅ Accuracy (准确率):  64.78%
🎯 Precision (查准率): 68.21%
🔎 Recall    (查全率): 55.36%
⚖️  F1-Score:         61.12%
----------------------------------------
详细混淆矩阵:
  [TP] 讽刺且预测对: 191
  [TN] 正常且预测对: 256
  [FP] 误判为讽刺  : 89 (模型太敏感)
  [FN] 漏判讽刺    : 154 (模型没识别出)
